In [1]:
import numpy as np
import pandas as pd
import Soil_Carbon_Model as md
import os
import openpyxl
from pandas import *

In [2]:
dt = 1 / 24
nyears = 1000
ndays = nyears * 365
nsim = ndays * 24
nnpp = ndays * 24

#Rainfall from Marked Poisson
rain = md.poisson_process(n = int(nsim), alpha = 14.42/1000, lbd = (0.19 * dt))

tsim = np.linspace(1, nnpp, int(nnpp))

days = np.arange(365)
annual_avg_npp = 456.3

shape = np.sin((np.pi*days)/365)                      

daily_npp = annual_avg_npp * (shape / np.sum(shape))
hourly_npp = np.repeat(daily_npp, 24)

npp = np.tile(hourly_npp,nyears)

In [3]:
print("Maximum Value is:", max(npp))
print("Average value is:", np.mean(npp))

Maximum Value is: 1.9637045245762004
Average value is: 1.2501369863013712


In [4]:
total_rainfall = rain.sum()                  # total rainfall in 'm'
print(f"Total rainfall over 450 years = {total_rainfall:.3f} meters")

RPY = (total_rainfall / nyears)*1000         # average rainfall per year in 'mm'
print(f"Average rainfall per year = {RPY:.3f} mm/year")

PET = 689.1
AI = PET/RPY
print(AI)

Total rainfall over 450 years = 1000.272 meters
Average rainfall per year = 1000.272 mm/year
0.6889123495643167


In [5]:
# Compute TI_DI and TI_Texture

KA_final = 9.0132
n_final = 1.7444
R_max = 3.8                   

def compute_TI(AI):
    AI = np.asarray(AI, dtype=float)
    numerator   = AI**n_final
    denominator = KA_final**n_final + AI**n_final
    norm_factor = R_max**n_final / (KA_final**n_final + R_max**n_final)
    TI = (numerator / denominator) / norm_factor

    return np.minimum(TI, 1.0)

# ------------------------------------
# TI FROM SAND (SIGMOID)
# ------------------------------------
sand = 21.0   # POI sand (%)
s50 = 25.0
k   = 0.08

def TI_from_sand(sand):
    return 1.0 / (1.0 + np.exp(-k * (sand - s50)))

TI_sand = TI_from_sand(sand)
TI_AI = compute_TI(AI)
TI = (TI_sand + TI_AI)/2

print("TI_AI =", TI_AI)
print("TI_sand =", TI_sand)
print("TI Total =", TI)

TI_AI = 0.06143281234172333
TI_sand = 0.42067574785125056
TI Total = 0.24105428009648694


In [6]:
#Hydraulic parameterization for dynamic porosity -------------------------------
alpha1 = 580  #hydraulic conductivity shape parameter
beta1 = 6     #hydraulic conductivity shape parameter
alpha2 = 580  #porosity shape parameter
beta2 = 6     #porosity shape parameter
kstex = 0.20   #m/d,
kstot = 1     #m/d
phitex = 0.45  
phitot = 0.60  

In [7]:
# Simulation for 2150 years at dynamic porosity
sim = md.run_julia(rain, npp, kstex, kstot, phitex, phitot, alpha1, beta1, alpha2, beta2, TI)
sim_year = sim.pivot_table(values=['s', 'phi', 'ksstr', 'P', 'DOC', 'A', 'M', 'B',
                                   'Fa', 'Fpa','Flb','Fl', 'Flm', 'Fpl', 'Fbm','Fld', 'Fal',
                                   'lk','npp_', 'L','Fi', 'Flpom', 'decomp'],
                           index='Years')

print(sim_year)

100%|██████████████████████████████████████████████████████████████████████| 8760000/8760000 [18:28<00:00, 7899.60it/s]


                 A           B         DOC        Fa       Fal       Fbm  \
Years                                                                      
1       884.015159   89.135304   30.178879  0.079028  0.002303  0.210431   
2       875.587879   43.562629   32.794679  0.076010  0.001801  0.102843   
3       935.674579   21.704753   33.466648  0.083319  0.001449  0.051241   
4       982.950950   10.924116   35.816149  0.086634  0.000990  0.025790   
5      1022.269367    5.545786   37.413822  0.090527  0.000622  0.013092   
...            ...         ...         ...       ...       ...       ...   
996    1769.340340  128.887267  191.802884  0.155811  0.004882  0.304253   
997    1769.472052  129.500300  192.382293  0.156618  0.004911  0.305701   
998    1769.543649  128.256117  186.840415  0.157092  0.004919  0.302765   
999    1769.575002  124.126198  187.922915  0.156237  0.004866  0.293015   
1000   1769.619254  123.202502  189.385503  0.157132  0.004888  0.290835   

           

In [8]:
sim_year.to_csv(r"C:\Users\rishabh.singh\OneDrive - Texas A&M AgriLife\Desktop\Slope\Poirier\POI.csv", index= True, header = True)
sim_year.to_csv(r"C:\Users\rishabh.singh\OneDrive - Texas A&M AgriLife\Desktop\Slope\Results\POI.csv", index= True, header = True)